# Домашнее задание HW02

## Анализ и очистка транзакционных данных

**Цель работы:** Провести первичный анализ датасета, выявить аномалии, очистить данные от ошибок и дубликатов, а также визуализировать ключевые метрики с помощью `matplotlib` и `seaborn`.

### Этапы работы:
1.  **Загрузка**: Импорт библиотек и чтение CSV.
2.  **Data Quality**: Поиск пропусков, дубликатов и логических ошибок.
3.  **Очистка**: Фильтрация данных на основе правил (возрастные ограничения, неотрицательные покупки).
4.  **EDA**: Группировка данных и расчет средних показателей.
5.  **Визуализация**: Построение гистограммы, боксплота и диаграммы рассеяния.

### 1. Импорт библиотек и настройка окружения

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Установка стиля графиков для эстетичности
plt.style.use('ggplot')
sns.set_context("notebook", font_scale=1.1)

# Создадим папку для графиков, если её нет
os.makedirs("figures", exist_ok=True)

### 2. Загрузка данных и первичный осмотр

In [ ]:
# Загружаем учебный датасет
try:
    raw_data = pd.read_csv("S02-hw-dataset.csv")
    print("Данные успешно загружены.")
except FileNotFoundError:
    print("Файл не найден. Проверьте путь.")

# Смотрим на размерность и типы данных
print(f"\nРазмерность датасета: {raw_data.shape}")
display(raw_data.head())

In [ ]:
print("--- Информация о типах данных ---")
raw_data.info()

print("\n--- Описательные статистики ---")
display(raw_data.describe().round(2))

### 3. Контроль качества данных (Data Quality)

In [ ]:
# 1. Проверка пропусков
missing_vals = raw_data.isna().sum()
print(f"Пропуски в данных:\n{missing_vals[missing_vals > 0]}")

# 2. Проверка дубликатов
duplicates_count = raw_data.duplicated().sum()
print(f"\nКоличество полных дубликатов строк: {duplicates_count}")

# 3. Поиск логических аномалий
# - Отрицательные покупки или выручка
# - Слишком маленький или нереалистично большой возраст
anomalies = raw_data[
    (raw_data['age'] < 10) | 
    (raw_data['age'] > 100) | 
    (raw_data['purchases'] < 0) | 
    (raw_data['revenue'] < 0)
]

print(f"\nНайдено подозрительных записей (аномалий): {len(anomalies)}")
display(anomalies)

### 4. Очистка данных

**Стратегия очистки:**
1. Удаляем полные дубликаты.
2. Исключаем записи с пропусками в возрасте (так как это ключевая метрика для анализа).
3. Фильтруем возраст: оставляем только диапазон 14–95 лет.
4. Убираем технические ошибки (отрицательные покупки).
5. Исключаем несогласованные данные (покупки есть, а выручка 0).

In [ ]:
# Создаем копию для очистки
df_clean = raw_data.copy()

# 1. Удаление дубликатов
df_clean = df_clean.drop_duplicates()

# 2. & 3. Фильтрация возраста (убираем NaN и выбросы)
age_filter = (df_clean['age'].notna()) & (df_clean['age'] >= 14) & (df_clean['age'] <= 95)
df_clean = df_clean[age_filter]

# 4. Логическая фильтрация метрик
metrics_filter = (df_clean['purchases'] >= 0) & (df_clean['revenue'] >= 0)
df_clean = df_clean[metrics_filter]

# 5. Проверка бизнес-логики: если покупок > 0, выручка должна быть > 0
inconsistent_data = (df_clean['purchases'] > 0) & (df_clean['revenue'] == 0)
df_clean = df_clean[~inconsistent_data]

# Результат очистки
print(f"Изначально строк: {raw_data.shape[0]}")
print(f"После очистки:    {df_clean.shape[0]}")
print(f"Удалено строк:    {raw_data.shape[0] - df_clean.shape[0]}")

### 5. Feature Engineering и Агрегация

In [ ]:
# Создадим категории по уровню дохода
df_clean['revenue_segment'] = pd.cut(
    df_clean['revenue'], 
    bins=[-1, 500, 1500, 10000], 
    labels=['Low (<500)', 'Medium (500-1500)', 'High (>1500)']
)

# Агрегируем статистику по странам
geo_stats = df_clean.groupby('country').agg({
    'revenue': ['mean', 'sum'],
    'purchases': ['mean', 'count'],
    'age': 'mean'
}).round(1)

print("Статистика по странам:")
# Используем раскраску для наглядности (чем темнее, тем больше значение)
display(geo_stats.style.background_gradient(cmap='Blues'))

### 6. Визуализация данных

Построим три графика для понимания распределений и зависимостей.

In [ ]:
# График 1: Гистограмма возраста
plt.figure(figsize=(10, 6))

plt.hist(df_clean['age'], bins=12, color='#48bf91', edgecolor='white', alpha=0.8)

plt.title('Распределение возраста пользователей (после очистки)', fontsize=14)
plt.xlabel('Возраст', fontsize=12)
plt.ylabel('Количество пользователей', fontsize=12)
plt.grid(axis='y', alpha=0.5)

# Сохранение
plt.savefig('figures/age_hist.png', dpi=300)
plt.show()

In [ ]:
# График 2: Boxplot распределения выручки по странам
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_clean, 
    x='revenue', 
    y='country',  # Горизонтальная ориентация
    palette='Set2',
    linewidth=1.5
)

plt.title('Анализ выручки в разрезе стран', fontsize=14)
plt.xlabel('Выручка', fontsize=12)
plt.ylabel('Страна', fontsize=12)

plt.savefig('figures/revenue_country_box.png', dpi=300)
plt.show()

In [ ]:
# График 3: Scatter plot (Возраст vs Выручка)
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df_clean, 
    x='age', 
    y='revenue', 
    hue='country', 
    style='country', 
    s=100, 
    alpha=0.8, 
    palette='deep'
)

plt.title('Зависимость выручки от возраста', fontsize=14)
plt.xlabel('Возраст', fontsize=12)
plt.ylabel('Выручка', fontsize=12)
plt.legend(title='Страна', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('figures/age_revenue_scatter.png', dpi=300)
plt.show()

#### Выводы по анализу:
1.  **Качество данных:** Исходный датасет содержал около 10-15% "мусорных" записей (дубликаты, некорректный возраст, отрицательные значения), которые были успешно отфильтрованы.
2.  **География:** Наибольшую активность показывают пользователи из России (RU) и Франции (FR).
3.  **Возраст:** Основная аудитория продукта — люди от 25 до 45 лет.
4.  **Доходность:** Германия (DE) показывает высокий средний чек, несмотря на меньшее количество пользователей в выборке.